#### act1_led.py

In [ ]:
# act1_led.py - blink the board's red LED
import time
import board
from digitalio import DigitalInOut, Direction

led = DigitalInOut(board.LED)
led.direction = Direction.OUTPUT

while True:
    led.value = True       # on
    time.sleep(0.5)
    led.value = False      # off
    time.sleep(0.5)

#### act2_buzzer.py

In [ ]:
# act2_buzzer.py - beep the active buzzer on D8 three times
import time
import board
from digitalio import DigitalInOut, Direction

buzzer = DigitalInOut(board.D8)
buzzer.direction = Direction.OUTPUT

for i in range(3):
    buzzer.value = True    # on: an active buzzer makes its own tone
    time.sleep(0.2)
    buzzer.value = False   # off
    time.sleep(0.3)


#### act3_sonar.py

In [ ]:
# act3_sonar.py - print the distance in cm (Thonny: View > Plotter)
import time
import board
import adafruit_hcsr04

sonar = adafruit_hcsr04.HCSR04(trigger_pin=board.D9, echo_pin=board.D2)

while True:
    try:
        print((sonar.distance,))
    except RuntimeError:
        print("no echo")   # nothing in range, or a soft surface
    time.sleep(0.1)

#### act4_motor.py

In [ ]:
# act4_motor.py - check each wheel turns the right way
# Lift the robot so the wheels spin freely.
import time
import board
import pwmio
from digitalio import DigitalInOut, Direction

left_dir = DigitalInOut(board.D4)        # M1 direction
left_dir.direction = Direction.OUTPUT
left_pwm = pwmio.PWMOut(board.D3)        # M1 speed

right_dir = DigitalInOut(board.D12)      # M2 direction
right_dir.direction = Direction.OUTPUT
right_pwm = pwmio.PWMOut(board.D11)      # M2 speed

SPEED = 40000                            # duty cycle: 0 = stop, 65535 = full

for name, d, pwm in (("LEFT", left_dir, left_pwm), ("RIGHT", right_dir, right_pwm)):
    print(name, "wheel should spin FORWARD")
    d.value = True                       # HIGH = forward on this robot
    pwm.duty_cycle = SPEED
    time.sleep(2)
    pwm.duty_cycle = 0
    time.sleep(1)

    print(name, "wheel should spin BACKWARD")
    d.value = False
    pwm.duty_cycle = SPEED
    time.sleep(2)
    pwm.duty_cycle = 0
    time.sleep(1)

print("done: if a wheel went the wrong way, swap that motor's two wires")

#### act5_square.py

In [ ]:
# act5_square.py
# Left motor on M1, right motor on M2, castor at the back.

import time
import board
import digitalio
import pwmio


class Motor:
    """One DC motor on the DFRobot shield: a PWM speed pin + a direction pin."""

    def __init__(self, pwm_pin, dir_pin, forward_level):
        self.pwm = pwmio.PWMOut(pwm_pin, frequency=1000, duty_cycle=0)
        self.dir = digitalio.DigitalInOut(dir_pin)
        self.dir.direction = digitalio.Direction.OUTPUT
        # The direction pin value (True or False) that makes this wheel go forward.
        # If the wheel turns the wrong way, swap True and False when you create it.
        self.forward_level = forward_level

    def run(self, speed):
        """speed from -1.0 (full reverse) to 1.0 (full forward); 0 = stop."""
        speed = max(-1.0, min(1.0, speed))          # keep within range [-1.0, 1.0]
        self.dir.value = self.forward_level if speed >= 0 else (not self.forward_level)
        self.pwm.duty_cycle = int(abs(speed) * 65535)

    def stop(self):
        self.pwm.duty_cycle = 0


# Set up the two wheels
left = Motor(board.D3, board.D4, forward_level=True)
right = Motor(board.D11, board.D12, forward_level=True)


# Robot movements
def drive(left_speed, right_speed):
    left.run(left_speed)
    right.run(right_speed)


def forward(speed=0.6):
    drive(speed, speed)


def backward(speed=0.6):
    drive(-speed, -speed)


def spin_left(speed=0.5):      
    drive(-speed, speed)


def spin_right(speed=0.5):
    drive(speed, -speed)


def stop():
    left.stop()
    right.stop()


# Demo: drive a square
try:
    time.sleep(2)              # time to put the robot down
    for side in range(4):
        print("Side", side + 1)
        forward(0.6)
        time.sleep(1)
        stop()
        time.sleep(0.3)
        spin_right(0.5)
        time.sleep(0.5)       # adjust until the turn is close to 90 degrees
        stop()
        time.sleep(0.3)
finally:
    stop()                     # motors always stop, even on Ctrl+C or an error
    print("Stopped")

#### code.py

In [ ]:
#import act1_led
#import act2_buzzer
#import act3_sonar
import act4_motor
#import act5_square